In [2]:
import torch
from itertools import combinations

In [3]:
def fit_ols_pairwise(X_ij: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    X = torch.cat(
        [
            torch.ones((X_ij.shape[0], 1), dtype=X_ij.dtype, device=X_ij.device),
            X_ij,
        ],
        dim=1,
    )
    XtX = X.T @ X
    Xty = X.T @ y
    beta =  torch.linalg.lstsq(XtX, Xty.unsqueeze(-1))
    return beta.solution.squeeze(-1)


def all_pairs_explicit_columns(X: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    pairs = list(combinations(range(X.shape[1]), 2))
    out = torch.empty((len(pairs), 3), dtype=X.dtype, device=X.device)
    for k, (i, j) in enumerate(pairs):
        out[k] = fit_ols_pairwise(X[:, [i, j]], y)
    return out


def apply_pairwise_coefficients(X: torch.Tensor, coefs: torch.Tensor) -> torch.Tensor:
    p = X.shape[1]
    pairs = list(combinations(range(p), 2))
    n, c = X.shape[0], len(pairs)
    results = torch.empty((n, c), dtype=X.dtype, device=X.device)
    for k, (i, j) in enumerate(pairs):
        b0, b1, b2 = coefs[k]
        results[:, k] = b0 + b1 * X[:, i] + b2 * X[:, j]
    return results


evaluate = lambda preds, y:  torch.mean(torch.abs(preds - y.unsqueeze(1)), dim=0)


def fit_all_pairs_vectorized(X: torch.Tensor, y: torch.Tensor) -> torch.Tensor:  
    G = X.T @ X
    g = X.T @ y
    sx = X.sum(dim=0)
    sy = y.sum()

    pairs = torch.tensor(list(combinations(range(X.shape[1]), 2)), dtype=torch.long)
    i, j = pairs[:, 0], pairs[:, 1]
    c = pairs.shape[0]

    A = torch.zeros((c, 3, 3), dtype=X.dtype)
    A[:, 0, 0] = float(X.shape[0],)
    A[:, 0, 1] = sx[i]
    A[:, 0, 2] = sx[j]
    A[:, 1, 0] = sx[i]
    A[:, 1, 1] = G[i, i]
    A[:, 1, 2] = G[i, j]
    A[:, 2, 0] = sx[j]
    A[:, 2, 1] = G[j, i]
    A[:, 2, 2] = G[j, j]

    b = torch.zeros((c, 3), dtype=X.dtype)
    b[:, 0] = sy
    b[:, 1] = g[i]
    b[:, 2] = g[j]

    betas = torch.linalg.lstsq(A, b)
    return betas.solution


def epoch(X, y, vectorized=False):
    if vectorized:
        coef = fit_all_pairs_vectorized(X, y)
    else:
        coef = all_pairs_explicit_columns(X, y)
        
    preds = apply_pairwise_coefficients(X, coef)
    errors = evaluate(preds, y)
    mask = torch.argsort(errors)
    print(errors.min())
    return preds[:, mask[:100]]

In [ ]:
X = torch.randn(5,3)
for i, j in combinations(range(X.shape[1]), 2):
    print(i, j)

0 1
0 2
1 2


# Explicit

In [4]:
X = torch.load("X_ep0.pt", weights_only=True)
y = torch.load("y_ep0.pt", weights_only=True)
X.shape, y.shape

(torch.Size([16512, 64]), torch.Size([16512]))

In [5]:
for _ in range(5):
    X = epoch(X, y)

tensor(0.5952)
tensor(0.5776)
tensor(0.5707)
tensor(0.5617)
tensor(0.5523)


# Vectorized learning

In [6]:
X = torch.load("X_ep0.pt", weights_only=False)
y = torch.load("y_ep0.pt", weights_only=False)
X.shape, y.shape

(torch.Size([16512, 64]), torch.Size([16512]))

In [7]:
for _ in range(5):
    X = epoch(X, y, vectorized=True)

tensor(0.5952)
tensor(0.5787)
tensor(0.5711)
tensor(0.5691)
tensor(0.5671)


# Vectorized learning & prediction

In [8]:
def apply_vectorized_coefficients(X: torch.Tensor, coefs: torch.Tensor) -> torch.Tensor:
  p = X.shape[1]
  pairs = torch.tensor(list(combinations(range(p), 2)), dtype=torch.long, device=X.device)
  i, j = pairs[:, 0], pairs[:, 1]
  b0, b1, b2 = coefs[:, 0], coefs[:, 1], coefs[:, 2]
  return b0.unsqueeze(0) + X[:, i] * b1.unsqueeze(0) + X[:, j] * b2.unsqueeze(0)

  
def epoch(X, y, vectorized=False):
    if vectorized:
        coef = fit_all_pairs_vectorized(X, y)
        preds = apply_vectorized_coefficients(X, coef)
    else:
        coef = all_pairs_explicit_columns(X, y)
        preds = apply_pairwise_coefficients(X, coef)
        
    errors = evaluate(preds, y)
    mask = torch.argsort(errors)
    print(errors.min())
    return preds[:, mask[:100]]

In [9]:
X = torch.load("X_ep0.pt", weights_only=False)
y = torch.load("y_ep0.pt", weights_only=False)
X.shape, y.shape

(torch.Size([16512, 64]), torch.Size([16512]))

In [10]:
for _ in range(5):
    X = epoch(X, y, vectorized=True)

tensor(0.5952)
tensor(0.5787)
tensor(0.5711)
tensor(0.5691)
tensor(0.5671)
